# Latent Video Watermarking dengan Neural Codec — Validated Pipeline V9

Pipeline V9 mengikuti alur proposal dosen secara langsung:

`Video → Neural Codec Encoder (g_a) → Latent Watermark Embedding → Quantization / Entropy Coding → Neural Codec Decoder (g_s) → Watermarked Video`

Watermark diekstraksi dengan mengubah video upload kembali ke ruang latent neural codec.
Identitas payload dilindungi Reed–Solomon dan CRC-16, sedangkan integritas isi video
dinilai secara terpisah menggunakan perceptual fingerprint.

V8 pixel-domain tetap berguna sebagai baseline, tetapi V9 adalah algoritma utama yang
menguji novelty penyisipan watermark pada latent feature neural codec.


## Penyesuaian terhadap arahan dosen

- Neural codec bukan lagi sekadar serangan kompresi; encoder dan decoder codec menjadi backbone watermarking.
- Embedder memodifikasi latent `y` sebelum quantization dan entropy coding.
- Extractor membaca watermark dari latent hasil re-encoding video yang diverifikasi.
- Payload enam karakter tetap dilindungi shortened Reed–Solomon RS(16,8) dan CRC-16.
- H.264, H.265/HEVC, AV1, dan neural recompression digunakan sebagai robustness attacks.
- PSNR/SSIM dilaporkan untuk kualitas keseluruhan dan penalti tambahan watermark terhadap neural baseline.
- Perceptual fingerprint memisahkan validasi integritas isi video dari validasi payload.
- Kasus video tampered disertakan agar sistem integritas mempunyai positive dan negative controls.
- Tool upload otomatis menampilkan watermark, CRC, presence, fingerprint distance, dan status integritas.

**Batasan:** backbone `bmshj2018_factorized` adalah learned image codec yang diterapkan
per frame. Jika dosen mewajibkan temporal neural video codec, backbone harus diganti,
sedangkan desain latent embedder, extractor, ECC, dan evaluasinya tetap dapat digunakan.


In [ ]:
# 1. Setup Colab
from google.colab import drive
drive.mount('/content/drive')

!pip install -q kagglehub opencv-python-headless scikit-image pandas matplotlib tqdm compressai reedsolo==1.7.0 pytorch-msssim==1.0.0 "gradio>=4.44,<7"
!apt-get -qq update
!apt-get -qq install -y ffmpeg

print('Dependency siap.')


In [ ]:
# 2. Import dan konfigurasi eksperimen
import binascii
import gc
import hashlib
import json
import math
import os
import random
import re
import struct
import subprocess
import tempfile
from collections import defaultdict
from pathlib import Path

import cv2
import gradio as gr
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_msssim import ssim as differentiable_ssim
from reedsolo import RSCodec, ReedSolomonError
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm.auto import tqdm

SEED = 42
FRAME_SIZE = (128, 128)
CLIP_FRAMES = 8
PAYLOADS_PER_STEP = 2
MAX_EVAL_FRAMES = 64

PAYLOAD_BYTES = 6
PAYLOAD_BITS = PAYLOAD_BYTES * 8
CRC_BYTES = 2
RS_PARITY_BYTES = 8
MESSAGE_BYTES = PAYLOAD_BYTES + CRC_BYTES
CODE_BYTES = MESSAGE_BYTES + RS_PARITY_BYTES
CODE_BITS = CODE_BYTES * 8

TARGET_TEXT = 'sabila'
CONTROL_TEXT = 'kontro'
CORE_NEURAL_QUALITY = 5
INTEGRITY_MAX_DISTANCE = 12

MAX_TRAIN_VIDEOS = 80
MAX_VAL_VIDEOS = 12
MAX_TEST_VIDEOS = 12

STEPS_PER_EPOCH = 50
VALIDATE_EVERY = 2
STAGE_PASS_PATIENCE = 2
RESET_V9_TRAINING = False

OUTPUT_ROOT = Path('/content/drive/MyDrive/Video_data/output/v9_latent_validated')
MODEL_DIR = OUTPUT_ROOT / 'models'
BITSTREAM_DIR = OUTPUT_ROOT / 'bitstreams'
REPORT_DIR = OUTPUT_ROOT / 'reports'
for directory in (OUTPUT_ROOT, MODEL_DIR, BITSTREAM_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

KAGGLE_DATASET_SLUG = 'abdallahwagih/ucf101-videos'
DATASET_ROOT = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
print('Dataset:', DATASET_ROOT)


In [ ]:
# 3. Discovery dan split anti-leakage
VIDEO_EXTENSIONS = {'.avi', '.mp4', '.mov', '.mkv'}
all_video_paths = sorted(
    path for path in DATASET_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
)
if not all_video_paths:
    raise FileNotFoundError(f'Tidak ada video di {DATASET_ROOT}')

def class_name(path):
    match = re.match(r'^v_(.+?)_g\d+_c\d+$', path.stem)
    return match.group(1) if match else path.parent.name

def group_key(path):
    match = re.search(r'_g(\d+)_', path.stem)
    group = match.group(1) if match else path.stem
    return f'{class_name(path)}:{group}'

def has_part(path, expected):
    return expected.lower() in {part.lower() for part in path.relative_to(DATASET_ROOT).parts}

def group_split(paths, fractions=(0.70, 0.15, 0.15), seed=SEED):
    groups = sorted({group_key(p) for p in paths})
    rng = random.Random(seed)
    rng.shuffle(groups)
    n = len(groups)
    n_train = max(1, int(n * fractions[0]))
    n_val = max(1, int(n * fractions[1]))
    train_groups = set(groups[:n_train])
    val_groups = set(groups[n_train:n_train + n_val])
    test_groups = set(groups[n_train + n_val:])
    return (
        [p for p in paths if group_key(p) in train_groups],
        [p for p in paths if group_key(p) in val_groups],
        [p for p in paths if group_key(p) in test_groups],
    )

explicit_train = [p for p in all_video_paths if has_part(p, 'train')]
explicit_test = [p for p in all_video_paths if has_part(p, 'test')]

if explicit_train and explicit_test:
    # Test bawaan dataset tidak pernah dipakai untuk training atau pemilihan checkpoint.
    train_candidates, val_candidates, _ = group_split(
        explicit_train, fractions=(0.80, 0.20, 0.0)
    )
    test_candidates = explicit_test
    split_source = 'folder train/test dataset + validation berbasis group dari train'
else:
    train_candidates, val_candidates, test_candidates = group_split(all_video_paths)
    split_source = 'group-aware split 70/15/15'

def balanced_sample(paths, limit, seed):
    by_class = defaultdict(list)
    for path in paths:
        by_class[class_name(path)].append(path)
    rng = random.Random(seed)
    for values in by_class.values():
        rng.shuffle(values)
    chosen = []
    classes = sorted(by_class)
    while len(chosen) < min(limit, len(paths)):
        progressed = False
        for cls in classes:
            if by_class[cls] and len(chosen) < limit:
                chosen.append(by_class[cls].pop())
                progressed = True
        if not progressed:
            break
    return chosen

TRAIN_PATHS = balanced_sample(train_candidates, MAX_TRAIN_VIDEOS, SEED + 1)
VAL_PATHS = balanced_sample(val_candidates, MAX_VAL_VIDEOS, SEED + 2)
TEST_PATHS = balanced_sample(test_candidates, MAX_TEST_VIDEOS, SEED + 3)

train_set, val_set, test_set = map(set, (TRAIN_PATHS, VAL_PATHS, TEST_PATHS))
assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)
train_groups = {group_key(p) for p in TRAIN_PATHS}
val_groups = {group_key(p) for p in VAL_PATHS}
test_groups = {group_key(p) for p in TEST_PATHS}
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print(f'Ditemukan: {len(all_video_paths)} video')
print('Sumber split:', split_source)
print(f'Train/Val/Test dipakai: {len(TRAIN_PATHS)}/{len(VAL_PATHS)}/{len(TEST_PATHS)}')
print('Kelas train:', sorted({class_name(p) for p in TRAIN_PATHS}))
print('Leakage check: LULUS')


In [ ]:
# 4. I/O video dengan frame count, resolusi, dan durasi yang konsisten
def _resize_rgb(frame_bgr, frame_size=FRAME_SIZE):
    frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (frame_size[1], frame_size[0]), interpolation=cv2.INTER_AREA)
    return frame.astype(np.float32) / 255.0

def sample_clip(path, clip_frames=CLIP_FRAMES):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise ValueError(f'Video tidak dapat dibaca: {path}')
    start = random.randint(0, max(total - clip_frames, 0))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(clip_frames):
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise ValueError(f'Tidak ada frame: {path}')
    while len(frames) < clip_frames:
        frames.append(frames[-1].copy())
    return np.stack(frames)

def load_eval_frames(path, max_frames=MAX_EVAL_FRAMES):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        raise ValueError(f'Video tidak dapat dibaca: {path}')
    indices = np.linspace(0, total - 1, min(max_frames, total), dtype=int)
    frames = []
    for index in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = cap.read()
        if ok:
            frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise ValueError(f'Tidak ada frame evaluasi: {path}')
    return np.stack(frames), float(fps)

def frames_to_tensor(frames):
    return torch.from_numpy(np.asarray(frames)).permute(0, 3, 1, 2).float().to(device)

def tensor_to_frames(tensor):
    return tensor.detach().clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

def read_encoded_video(path, expected_frames=None):
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(_resize_rgb(frame))
    cap.release()
    if not frames:
        raise RuntimeError(f'FFmpeg tidak menghasilkan frame terbaca: {path}')
    if expected_frames is not None:
        frames = frames[:expected_frames]
        while len(frames) < expected_frames:
            frames.append(frames[-1].copy())
    return np.stack(frames)

def ffmpeg_encode(frames, output_path, fps, codec, crf):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames_u8 = np.clip(np.asarray(frames) * 255.0, 0, 255).astype(np.uint8)
    height, width = frames_u8.shape[1:3]
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{width}x{height}',
        '-r', str(fps), '-i', '-', '-an', '-c:v', codec,
    ]
    if codec == 'libaom-av1':
        cmd += ['-crf', str(crf), '-b:v', '0', '-cpu-used', '8', '-row-mt', '1']
    else:
        cmd += ['-crf', str(crf), '-preset', 'fast']
    cmd += ['-pix_fmt', 'yuv420p', str(output_path)]
    result = subprocess.run(cmd, input=frames_u8.tobytes(), capture_output=True)
    if result.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(result.stderr.decode(errors='replace'))
    return output_path
print('Video I/O siap, termasuk AV1.')


In [ ]:
# 5. Payload, RS/CRC, latent embedder, dan latent extractor
RS_CODEC = RSCodec(RS_PARITY_BYTES)

def text_to_payload(text):
    raw = text.encode('ascii')
    if len(raw) != PAYLOAD_BYTES:
        raise ValueError(f'Teks harus tepat {PAYLOAD_BYTES} karakter ASCII')
    bits = np.unpackbits(np.frombuffer(raw, dtype=np.uint8), bitorder='big')
    return torch.tensor(bits, dtype=torch.float32, device=device).unsqueeze(0)

def payload_to_text(payload_bits):
    bits = np.asarray(payload_bits, dtype=np.uint8).reshape(-1)[:PAYLOAD_BITS]
    raw = np.packbits(bits, bitorder='big').tobytes()[:PAYLOAD_BYTES]
    return raw.decode('ascii', errors='replace')

def crc16(payload_bytes):
    return binascii.crc_hqx(payload_bytes, 0xFFFF)

def payload_to_codeword(payload):
    payload_np = payload.detach().round().to(torch.uint8).cpu().numpy()
    rows = []
    for row in payload_np:
        payload_bytes = np.packbits(row[:PAYLOAD_BITS], bitorder='big').tobytes()
        checksum = crc16(payload_bytes).to_bytes(CRC_BYTES, 'big')
        encoded = bytes(RS_CODEC.encode(payload_bytes + checksum))
        if len(encoded) != CODE_BYTES:
            raise RuntimeError(f'Panjang RS codeword {len(encoded)} != {CODE_BYTES}')
        rows.append(np.unpackbits(np.frombuffer(encoded, dtype=np.uint8), bitorder='big'))
    return torch.tensor(np.stack(rows), dtype=torch.float32, device=device)

def decode_codeword(codeword_bits):
    bits = np.asarray(codeword_bits, dtype=np.uint8).reshape(-1)[:CODE_BITS]
    raw_codeword = np.packbits(bits, bitorder='big').tobytes()[:CODE_BYTES]
    raw_payload = np.unpackbits(
        np.frombuffer(raw_codeword[:PAYLOAD_BYTES], dtype=np.uint8), bitorder='big'
    ).astype(np.uint8)
    result = {
        'ecc_success': False,
        'crc_valid': False,
        'payload_bits': raw_payload,
        'raw_payload_bits': raw_payload,
        'decoded_text': '<invalid>',
        'raw_text': payload_to_text(raw_payload),
        'corrected_symbols': np.nan,
    }
    try:
        decoded = RS_CODEC.decode(bytearray(raw_codeword))
        message = bytes(decoded[0] if isinstance(decoded, tuple) else decoded)
        errata = decoded[2] if isinstance(decoded, tuple) and len(decoded) > 2 else []
        if len(message) < MESSAGE_BYTES:
            return result
        payload_bytes = message[:PAYLOAD_BYTES]
        stored_crc = int.from_bytes(message[PAYLOAD_BYTES:MESSAGE_BYTES], 'big')
        corrected_payload = np.unpackbits(
            np.frombuffer(payload_bytes, dtype=np.uint8), bitorder='big'
        ).astype(np.uint8)
        crc_valid = stored_crc == crc16(payload_bytes)
        result.update({
            'ecc_success': True,
            'crc_valid': bool(crc_valid),
            'payload_bits': corrected_payload,
            'decoded_text': payload_bytes.decode('ascii', errors='replace') if crc_valid else '<invalid-crc>',
            'corrected_symbols': int(len(errata)),
        })
    except (ReedSolomonError, ValueError, IndexError):
        pass
    return result

def aggregate_decode(code_logits, presence_logits):
    probabilities = torch.sigmoid(code_logits).mean(dim=0)
    bits = (probabilities >= 0.5).to(torch.uint8)
    presence = torch.sigmoid(presence_logits).mean().item()
    return bits.cpu().numpy(), probabilities.cpu().numpy(), float(presence)

def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1

class LatentWatermarkEmbedder(nn.Module):
    def __init__(self, latent_channels, code_bits=CODE_BITS, grid_size=8, strength=1.50):
        super().__init__()
        self.latent_channels = latent_channels
        self.grid_size = grid_size
        self.strength = strength
        self.project = nn.Linear(code_bits, latent_channels * grid_size * grid_size)
        self.fusion = nn.Sequential(
            nn.Conv2d(latent_channels * 2, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
            nn.GroupNorm(group_count(latent_channels), latent_channels), nn.SiLU(),
            nn.Conv2d(latent_channels, latent_channels, 3, padding=1),
        )

    def forward(self, latent, codeword):
        message = self.project(codeword).view(
            latent.shape[0], self.latent_channels, self.grid_size, self.grid_size
        )
        if message.shape[-2:] != latent.shape[-2:]:
            message = F.interpolate(message, size=latent.shape[-2:], mode='bilinear', align_corners=False)
        delta = torch.tanh(self.fusion(torch.cat([latent, message], dim=1)))
        return latent + self.strength * delta, delta

class LatentWatermarkExtractor(nn.Module):
    def __init__(self, latent_channels, code_bits=CODE_BITS):
        super().__init__()
        hidden = max(192, latent_channels)
        self.features = nn.Sequential(
            nn.Conv2d(latent_channels, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.GroupNorm(group_count(hidden), hidden), nn.SiLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.shared = nn.Sequential(
            nn.Flatten(), nn.Linear(hidden * 4 * 4, 512), nn.SiLU(), nn.Dropout(0.02)
        )
        self.code_head = nn.Linear(512, code_bits)
        self.presence_head = nn.Linear(512, 1)

    def encode_features(self, latent):
        return self.shared(self.features(latent))

    def classify(self, hidden):
        return self.code_head(hidden), self.presence_head(hidden).squeeze(1)

    def forward(self, latent):
        return self.classify(self.encode_features(latent))

target_payload = text_to_payload(TARGET_TEXT)
target_codeword = payload_to_codeword(target_payload)
assert decode_codeword(target_codeword.cpu().numpy())['decoded_text'] == TARGET_TEXT
print(f'Payload {PAYLOAD_BITS} bit → RS/CRC codeword {CODE_BITS} bit.')


In [ ]:
# 6. Neural codec backbone, latent embedding, dan entropy-coded bitstream
from compressai.zoo import bmshj2018_factorized

_neural_models = {}

def get_neural_model(quality):
    quality = int(quality)
    if quality not in _neural_models:
        model = bmshj2018_factorized(
            quality=quality, pretrained=True
        ).to(device).eval()
        model.update(force=True)
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        _neural_models[quality] = model
    return _neural_models[quality]

CORE_CODEC = get_neural_model(CORE_NEURAL_QUALITY)
with torch.inference_mode():
    probe_latent = CORE_CODEC.g_a(torch.zeros(1, 3, *FRAME_SIZE, device=device))
LATENT_CHANNELS = int(probe_latent.shape[1])
LATENT_SHAPE = tuple(map(int, probe_latent.shape[-2:]))

latent_embedder = LatentWatermarkEmbedder(LATENT_CHANNELS).to(device)
latent_extractor = LatentWatermarkExtractor(LATENT_CHANNELS).to(device)

def quantize_latent_training(latent, noise=None):
    if noise is None:
        noise = torch.empty_like(latent).uniform_(-0.5, 0.5)
    return latent + noise, noise

def quantize_latent_eval(model, latent):
    medians = model.entropy_bottleneck._get_medians()
    return model.entropy_bottleneck.quantize(latent, 'dequantize', medians)

def latent_training_forward(frames, frame_codeword):
    latent = CORE_CODEC.g_a(frames)
    watermarked_latent, delta = latent_embedder(latent, frame_codeword)
    noise = torch.empty_like(latent).uniform_(-0.5, 0.5)
    baseline_hat, _ = quantize_latent_training(latent, noise)
    watermarked_hat, _ = quantize_latent_training(watermarked_latent, noise)
    baseline_frames = CORE_CODEC.g_s(baseline_hat).clamp(0, 1)
    watermarked_frames = CORE_CODEC.g_s(watermarked_hat).clamp(0, 1)
    return baseline_frames, watermarked_frames, latent, watermarked_latent, delta

def latent_eval_forward(frames, frame_codeword):
    latent = CORE_CODEC.g_a(frames)
    watermarked_latent, delta = latent_embedder(latent, frame_codeword)
    baseline_hat = quantize_latent_eval(CORE_CODEC, latent)
    watermarked_hat = quantize_latent_eval(CORE_CODEC, watermarked_latent)
    baseline_frames = CORE_CODEC.g_s(baseline_hat).clamp(0, 1)
    watermarked_frames = CORE_CODEC.g_s(watermarked_hat).clamp(0, 1)
    return baseline_frames, watermarked_frames, latent, watermarked_latent, delta

def plain_neural_encode(frames, output_path, fps, quality):
    model = get_neural_model(quality)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    entries = []
    with torch.inference_mode():
        for start in range(0, len(frames), 8):
            x = frames_to_tensor(frames[start:start + 8])
            packed = model.compress(x)
            shape = tuple(map(int, packed['shape']))
            for batch_index in range(x.shape[0]):
                streams = [group[batch_index] for group in packed['strings']]
                entries.append((shape, streams))
    header = json.dumps({
        'codec': 'bmshj2018-factorized', 'quality': int(quality),
        'fps': float(fps), 'frames': len(entries),
        'height': int(frames.shape[1]), 'width': int(frames.shape[2]),
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'NVC1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, streams in entries:
            handle.write(struct.pack('<IIH', shape[0], shape[1], len(streams)))
            for stream in streams:
                handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return output_path

def plain_neural_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'NVC1':
            raise ValueError('Bukan container NVC1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        model = get_neural_model(header['quality'])
        entries = []
        for _ in range(header['frames']):
            h, w, n_streams = struct.unpack('<IIH', handle.read(10))
            streams = []
            for _ in range(n_streams):
                size = struct.unpack('<I', handle.read(4))[0]
                streams.append(handle.read(size))
            entries.append(((h, w), streams))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), 8):
            batch = entries[start:start + 8]
            shape = batch[0][0]
            n_streams = len(batch[0][1])
            strings = [[entry[1][i] for entry in batch] for i in range(n_streams)]
            frames.extend(tensor_to_frames(model.decompress(strings, shape)['x_hat']))
    return np.stack(frames), header

def latent_watermark_encode(frames, output_path, fps, payload):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    codeword = payload_to_codeword(payload)
    entries = []
    reconstructed = []
    with torch.inference_mode():
        for start in range(0, len(frames), 8):
            x = frames_to_tensor(frames[start:start + 8])
            frame_codeword = codeword.repeat(x.shape[0], 1)
            latent = CORE_CODEC.g_a(x)
            watermarked_latent, _ = latent_embedder(latent, frame_codeword)
            strings = CORE_CODEC.entropy_bottleneck.compress(watermarked_latent)
            shape = tuple(map(int, watermarked_latent.shape[-2:]))
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            reconstructed.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
            entries.extend((shape, stream) for stream in strings)
    header = json.dumps({
        'codec': 'bmshj2018-factorized-latent-watermark',
        'quality': CORE_NEURAL_QUALITY, 'fps': float(fps),
        'frames': len(entries), 'height': int(frames.shape[1]),
        'width': int(frames.shape[2]), 'code_bits': CODE_BITS,
    }).encode('utf-8')
    with output_path.open('wb') as handle:
        handle.write(b'LWM1'); handle.write(struct.pack('<I', len(header))); handle.write(header)
        for shape, stream in entries:
            handle.write(struct.pack('<II', shape[0], shape[1]))
            handle.write(struct.pack('<I', len(stream))); handle.write(stream)
    return np.stack(reconstructed), output_path

def latent_watermark_decode(input_path):
    input_path = Path(input_path)
    with input_path.open('rb') as handle:
        if handle.read(4) != b'LWM1':
            raise ValueError('Bukan container latent watermark LWM1')
        header_len = struct.unpack('<I', handle.read(4))[0]
        header = json.loads(handle.read(header_len).decode('utf-8'))
        entries = []
        for _ in range(header['frames']):
            h, w = struct.unpack('<II', handle.read(8))
            size = struct.unpack('<I', handle.read(4))[0]
            entries.append(((h, w), handle.read(size)))
    frames = []
    with torch.inference_mode():
        for start in range(0, len(entries), 8):
            batch = entries[start:start + 8]
            shape = batch[0][0]
            strings = [entry[1] for entry in batch]
            latent_hat = CORE_CODEC.entropy_bottleneck.decompress(strings, shape)
            frames.extend(tensor_to_frames(CORE_CODEC.g_s(latent_hat).clamp(0, 1)))
    return np.stack(frames), header

print(
    f'Core neural quality={CORE_NEURAL_QUALITY}; latent={LATENT_CHANNELS}x'
    f'{LATENT_SHAPE[0]}x{LATENT_SHAPE[1]}; latent embedder siap.'
)


In [ ]:
# 7. Robustness attacks dan bounded latent curriculum V9
def ffmpeg_bpda(x, codec, crf, fps=25.0):
    with tempfile.TemporaryDirectory() as tmp:
        suffix = '.mkv' if codec == 'libaom-av1' else '.mp4'
        path = Path(tmp) / f'roundtrip{suffix}'
        frames = tensor_to_frames(x)
        ffmpeg_encode(frames, path, fps, codec, crf)
        reconstructed = frames_to_tensor(read_encoded_video(path, len(frames)))
    return x + (reconstructed - x).detach()

def neural_attack_differentiable(x, quality):
    model = get_neural_model(quality)
    if torch.is_grad_enabled():
        previous_mode = model.training
        model.train()
        reconstructed = model(x.clamp(0, 1))['x_hat'].clamp(0, 1)
        model.train(previous_mode)
        return reconstructed
    model.eval()
    return model(x.clamp(0, 1))['x_hat'].clamp(0, 1)

def apply_attack(x, specification):
    name, level = specification
    if name == 'identity':
        return x
    if name == 'noise':
        noisy = x + torch.randn_like(x) * level
        quantized = torch.clamp(torch.round(noisy * 255) / 255, 0, 1)
        return x + (quantized - x).detach()
    if name == 'blur':
        return F.avg_pool2d(x, 3, stride=1, padding=1)
    if name == 'resize':
        reduced = F.interpolate(x, scale_factor=float(level), mode='bilinear', align_corners=False)
        return F.interpolate(reduced, size=x.shape[-2:], mode='bilinear', align_corners=False)
    if name == 'h264':
        return ffmpeg_bpda(x, 'libx264', int(level))
    if name == 'h265':
        return ffmpeg_bpda(x, 'libx265', int(level))
    if name == 'av1':
        return ffmpeg_bpda(x, 'libaom-av1', int(level))
    if name == 'neural':
        return neural_attack_differentiable(x, int(level))
    raise ValueError(specification)

def latent_clip_prediction_tensor(frames, clips=1):
    observed_latent = CORE_CODEC.g_a(frames)
    hidden = latent_extractor.encode_features(observed_latent)
    hidden = hidden.view(clips, -1, hidden.shape[-1]).mean(dim=1)
    return latent_extractor.classify(hidden)

TRAINING_STAGES = [
    {
        'name': 'latent_payload_pretrain', 'max_epochs': 110, 'min_epochs': 20,
        'strength_start': 1.50, 'strength_end': 1.20, 'lr': 8e-4,
        'attacks': [('identity', 0), ('identity', 0), ('noise', 0.005), ('blur', 0)],
        'validation_attacks': [('identity', 0), ('noise', 0.005)],
        'presence_weight': 0.10, 'quality_target_mse': 2e-3,
        'quality_target_ssim': 0.85, 'quality_weight': 0.15,
        'min_raw_code_bit_acc': 0.960, 'min_post_ecc_exact': 0.80,
        'min_crc_valid': 0.80, 'min_presence': 0.80, 'min_negative': 0.80,
        'min_incremental_psnr': 26.0, 'min_incremental_ssim': 0.85,
    },
    {
        'name': 'latent_imperceptibility', 'max_epochs': 50, 'min_epochs': 12,
        'strength_start': 1.20, 'strength_end': 0.70, 'lr': 4e-4,
        'attacks': [('identity', 0), ('noise', 0.01), ('blur', 0), ('resize', 0.75)],
        'validation_attacks': [('identity', 0), ('noise', 0.01), ('resize', 0.75)],
        'presence_weight': 0.50, 'quality_target_mse': 3e-4,
        'quality_target_ssim': 0.95, 'quality_weight': 1.00,
        'min_raw_code_bit_acc': 0.930, 'min_post_ecc_exact': 0.80,
        'min_crc_valid': 0.80, 'min_presence': 0.95, 'min_negative': 0.95,
        'min_incremental_psnr': 34.0, 'min_incremental_ssim': 0.94,
    },
    {
        'name': 'modern_compression', 'max_epochs': 50, 'min_epochs': 12,
        'strength_start': 0.90, 'strength_end': 0.70, 'lr': 3e-4,
        'attacks': [
            ('noise', 0.015), ('h264', 28), ('h265', 28), ('av1', 28),
            ('neural', 5), ('neural', 3),
        ],
        'validation_attacks': [('h265', 28), ('av1', 28), ('neural', 5)],
        'presence_weight': 0.50, 'quality_target_mse': 3e-4,
        'quality_target_ssim': 0.95, 'quality_weight': 1.00,
        'min_raw_code_bit_acc': 0.880, 'min_post_ecc_exact': 0.75,
        'min_crc_valid': 0.75, 'min_presence': 0.90, 'min_negative': 0.95,
        'min_incremental_psnr': 34.0, 'min_incremental_ssim': 0.94,
    },
    {
        'name': 'high_compression', 'max_epochs': 60, 'min_epochs': 15,
        'strength_start': 1.00, 'strength_end': 0.75, 'lr': 2e-4,
        'attacks': [
            ('h264', 35), ('h264', 40), ('h265', 35), ('h265', 40),
            ('av1', 35), ('av1', 40), ('neural', 3), ('neural', 1), ('neural', 1),
        ],
        'validation_attacks': [('h265', 40), ('av1', 40), ('neural', 1)],
        'presence_weight': 0.50, 'quality_target_mse': 3e-4,
        'quality_target_ssim': 0.95, 'quality_weight': 1.25,
        'min_raw_code_bit_acc': 0.850, 'min_post_ecc_exact': 0.80,
        'min_crc_valid': 0.80, 'min_presence': 0.90, 'min_negative': 0.95,
        'min_incremental_psnr': 34.0, 'min_incremental_ssim': 0.94,
    },
]

def stage_passed(stage, metrics):
    return (
        metrics['raw_code_bit_acc'] >= stage['min_raw_code_bit_acc'] and
        metrics['post_ecc_exact_recovery'] >= stage['min_post_ecc_exact'] and
        metrics['crc_valid_rate'] >= stage['min_crc_valid'] and
        metrics['presence_tpr'] >= stage['min_presence'] and
        metrics['negative_rejection'] >= stage['min_negative'] and
        metrics['incremental_watermark_psnr'] >= stage['min_incremental_psnr'] and
        metrics['incremental_watermark_ssim'] >= stage['min_incremental_ssim']
    )

def quick_validation(stage, max_videos=4):
    latent_embedder.eval(); latent_extractor.eval()
    correct = total = exact = crc_count = samples = 0
    presences, rejections = [], []
    incremental_psnr, incremental_ssim, overall_psnr = [], [], []
    with torch.no_grad():
        for path in VAL_PATHS[:max_videos]:
            original = frames_to_tensor(sample_clip(path))
            payload = torch.randint(0, 2, (1, PAYLOAD_BITS), device=device).float()
            codeword = payload_to_codeword(payload)
            frame_codeword = codeword.repeat(original.shape[0], 1)
            baseline, watermarked, _, _, _ = latent_eval_forward(original, frame_codeword)
            inc_mse = F.mse_loss(watermarked, baseline).item()
            incremental_psnr.append(-10.0 * math.log10(max(inc_mse, 1e-12)))
            incremental_ssim.append(differentiable_ssim(
                watermarked, baseline, data_range=1.0, size_average=True
            ).item())
            overall_mse = F.mse_loss(watermarked, original).item()
            overall_psnr.append(-10.0 * math.log10(max(overall_mse, 1e-12)))
            for attack in stage['validation_attacks']:
                positive = apply_attack(watermarked, attack)
                negative = apply_attack(baseline, attack)
                logits, presence_logits = latent_clip_prediction_tensor(positive)
                _, negative_presence = latent_clip_prediction_tensor(negative)
                predicted, _, presence = aggregate_decode(logits, presence_logits)
                outcome = decode_codeword(predicted)
                expected_code = codeword.cpu().numpy().reshape(-1).astype(np.uint8)
                target = payload.cpu().numpy().reshape(-1).astype(np.uint8)
                correct += int((predicted == expected_code).sum()); total += CODE_BITS
                exact += int(outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], target))
                crc_count += int(outcome['crc_valid']); samples += 1
                presences.append(presence >= 0.5)
                rejections.append(torch.sigmoid(negative_presence).mean().item() < 0.5)
    latent_embedder.train(); latent_extractor.train()
    return {
        'raw_code_bit_acc': correct / max(total, 1),
        'post_ecc_exact_recovery': exact / max(samples, 1),
        'crc_valid_rate': crc_count / max(samples, 1),
        'presence_tpr': float(np.mean(presences)),
        'negative_rejection': float(np.mean(rejections)),
        'incremental_watermark_psnr': float(np.mean(incremental_psnr)),
        'incremental_watermark_ssim': float(np.mean(incremental_ssim)),
        'overall_original_to_watermarked_psnr': float(np.mean(overall_psnr)),
    }

print('Latent curriculum V9:', [(stage['name'], stage['max_epochs']) for stage in TRAINING_STAGES])


In [ ]:
# 8. Training V9 — latent embedding, clip extraction, robustness, dan imperceptibility
config_for_hash = {
    'frame_size': FRAME_SIZE, 'payload_bits': PAYLOAD_BITS, 'code_bits': CODE_BITS,
    'core_neural_quality': CORE_NEURAL_QUALITY, 'latent_channels': LATENT_CHANNELS,
    'clip_frames': CLIP_FRAMES, 'payloads_per_step': PAYLOADS_PER_STEP,
    'seed': SEED, 'version': 9, 'curriculum': TRAINING_STAGES,
}
config_hash = hashlib.sha256(json.dumps(
    config_for_hash, sort_keys=True
).encode()).hexdigest()[:10]
checkpoint_path = MODEL_DIR / f'checkpoint_v9_{config_hash}.pth'

def make_optimizer(stage):
    return torch.optim.AdamW(
        list(latent_embedder.parameters()) + list(latent_extractor.parameters()),
        lr=stage['lr'], weight_decay=1e-5,
    )

def model_snapshot(stage_index, stage_epoch, validation):
    return {
        'latent_embedder': latent_embedder.state_dict(),
        'latent_extractor': latent_extractor.state_dict(),
        'latent_strength': latent_embedder.strength,
        'config_hash': config_hash, 'global_epoch': global_epoch,
        'stage_index': stage_index, 'stage_epoch': stage_epoch,
        'validation': validation,
    }

def save_checkpoint(optimizer):
    torch.save({
        'latent_embedder': latent_embedder.state_dict(),
        'latent_extractor': latent_extractor.state_dict(),
        'optimizer': None if optimizer is None else optimizer.state_dict(),
        'latent_strength': latent_embedder.strength,
        'config_hash': config_hash, 'global_epoch': global_epoch,
        'current_stage': current_stage, 'stage_epoch': stage_epoch,
        'stage_best_score': stage_best_score, 'stage_pass_streak': stage_pass_streak,
        'history': history, 'stage_records': stage_records,
    }, checkpoint_path)

current_stage = stage_epoch = global_epoch = stage_pass_streak = 0
stage_best_score = -math.inf
history, stage_records = [], []
resume_optimizer_state = None

if RESET_V9_TRAINING:
    if checkpoint_path.exists(): checkpoint_path.unlink()
    for stale in MODEL_DIR.glob(f'best_v9_stage*_{config_hash}.pth'):
        stale.unlink()
    print('Checkpoint dan best model V9 lama dibersihkan.')
elif checkpoint_path.exists():
    saved = torch.load(checkpoint_path, map_location=device)
    if saved.get('config_hash') != config_hash:
        raise RuntimeError('Checkpoint V9 tidak cocok dengan konfigurasi.')
    latent_embedder.load_state_dict(saved['latent_embedder'])
    latent_extractor.load_state_dict(saved['latent_extractor'])
    latent_embedder.strength = saved.get('latent_strength', latent_embedder.strength)
    current_stage = saved.get('current_stage', 0)
    stage_epoch = saved.get('stage_epoch', 0)
    global_epoch = saved.get('global_epoch', 0)
    stage_best_score = saved.get('stage_best_score', -math.inf)
    stage_pass_streak = saved.get('stage_pass_streak', 0)
    history = saved.get('history', [])
    stage_records = saved.get('stage_records', [])
    resume_optimizer_state = saved.get('optimizer')
    print(f'Resume V9: global epoch {global_epoch}, stage {current_stage + 1}.')

optimizer = None
optimizer_stage = None
while current_stage < len(TRAINING_STAGES):
    stage = TRAINING_STAGES[current_stage]
    if optimizer is None or optimizer_stage != current_stage:
        optimizer = make_optimizer(stage); optimizer_stage = current_stage
        if resume_optimizer_state is not None:
            optimizer.load_state_dict(resume_optimizer_state)
            resume_optimizer_state = None

    progress = stage_epoch / max(stage['max_epochs'] - 1, 1)
    latent_embedder.strength = (
        stage['strength_start'] +
        (stage['strength_end'] - stage['strength_start']) * progress
    )
    stage_epoch += 1; global_epoch += 1
    latent_embedder.train(); latent_extractor.train()
    totals = defaultdict(float)

    for _ in tqdm(
        range(STEPS_PER_EPOCH),
        desc=f'Global {global_epoch} | {stage["name"]} {stage_epoch}/{stage["max_epochs"]}',
        leave=False,
    ):
        base_clip = frames_to_tensor(sample_clip(random.choice(TRAIN_PATHS)))
        frames_per_payload = base_clip.shape[0]
        payload = torch.randint(0, 2, (PAYLOADS_PER_STEP, PAYLOAD_BITS), device=device).float()
        clip_codeword = payload_to_codeword(payload)
        original = base_clip.repeat(PAYLOADS_PER_STEP, 1, 1, 1)
        frame_codeword = clip_codeword.repeat_interleave(frames_per_payload, dim=0)

        baseline, watermarked, _, _, delta = latent_training_forward(original, frame_codeword)
        attack = random.choice(stage['attacks'])
        positive = apply_attack(watermarked, attack)
        negative = apply_attack(baseline, attack)

        positive_latent = CORE_CODEC.g_a(positive)
        negative_latent = CORE_CODEC.g_a(negative)
        positive_hidden = latent_extractor.encode_features(positive_latent)
        negative_hidden = latent_extractor.encode_features(negative_latent)
        frame_bits, frame_presence = latent_extractor.classify(positive_hidden)
        _, negative_frame_presence = latent_extractor.classify(negative_hidden)
        positive_clip_hidden = positive_hidden.view(
            PAYLOADS_PER_STEP, frames_per_payload, -1
        ).mean(dim=1)
        negative_clip_hidden = negative_hidden.view(
            PAYLOADS_PER_STEP, frames_per_payload, -1
        ).mean(dim=1)
        clip_bits, clip_presence = latent_extractor.classify(positive_clip_hidden)
        _, negative_clip_presence = latent_extractor.classify(negative_clip_hidden)

        payload_loss = (
            0.35 * F.binary_cross_entropy_with_logits(frame_bits, frame_codeword) +
            1.65 * F.binary_cross_entropy_with_logits(clip_bits, clip_codeword)
        )
        presence_loss = 0.25 * (
            F.binary_cross_entropy_with_logits(frame_presence, torch.ones_like(frame_presence)) +
            F.binary_cross_entropy_with_logits(
                negative_frame_presence, torch.zeros_like(negative_frame_presence)
            ) +
            F.binary_cross_entropy_with_logits(clip_presence, torch.ones_like(clip_presence)) +
            F.binary_cross_entropy_with_logits(
                negative_clip_presence, torch.zeros_like(negative_clip_presence)
            )
        )

        incremental_mse = F.mse_loss(watermarked, baseline)
        incremental_ssim = differentiable_ssim(
            watermarked, baseline, data_range=1.0, size_average=True
        )
        mse_penalty = F.relu(
            incremental_mse - stage['quality_target_mse']
        ) / stage['quality_target_mse']
        ssim_penalty = F.relu(
            stage['quality_target_ssim'] - incremental_ssim
        ) / max(1.0 - stage['quality_target_ssim'], 0.05)
        quality_loss = mse_penalty + ssim_penalty
        latent_energy = delta.pow(2).mean()
        loss = (
            2.0 * payload_loss + stage['presence_weight'] * presence_loss +
            stage['quality_weight'] * quality_loss + 0.01 * latent_energy
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(latent_embedder.parameters()) + list(latent_extractor.parameters()), 5.0
        )
        optimizer.step()

        with torch.no_grad():
            totals['loss'] += loss.item()
            totals['raw_code_bit_acc'] += (
                (clip_bits > 0) == clip_codeword.bool()
            ).float().mean().item()
            totals['presence_tpr'] += float(torch.sigmoid(clip_presence).mean() >= 0.5)
            totals['negative_rejection'] += float(
                torch.sigmoid(negative_clip_presence).mean() < 0.5
            )
            totals['incremental_mse'] += incremental_mse.item()
            totals['incremental_ssim'] += incremental_ssim.item()
            totals['overall_mse'] += F.mse_loss(watermarked, original).item()

    row = {
        'global_epoch': global_epoch, 'stage_index': current_stage + 1,
        'stage_name': stage['name'], 'stage_epoch': stage_epoch,
        'stage_max_epochs': stage['max_epochs'],
        'latent_strength': latent_embedder.strength,
        'loss': totals['loss'] / STEPS_PER_EPOCH,
        'train_raw_code_bit_acc': totals['raw_code_bit_acc'] / STEPS_PER_EPOCH,
        'train_presence_tpr': totals['presence_tpr'] / STEPS_PER_EPOCH,
        'train_negative_rejection': totals['negative_rejection'] / STEPS_PER_EPOCH,
        'train_incremental_psnr': -10.0 * math.log10(max(
            totals['incremental_mse'] / STEPS_PER_EPOCH, 1e-12
        )),
        'train_incremental_ssim': totals['incremental_ssim'] / STEPS_PER_EPOCH,
        'train_overall_psnr': -10.0 * math.log10(max(
            totals['overall_mse'] / STEPS_PER_EPOCH, 1e-12
        )),
    }

    transition_reason = None
    if global_epoch % VALIDATE_EVERY == 0 or stage_epoch == stage['max_epochs']:
        validation = quick_validation(stage)
        row.update({f'val_{key}': value for key, value in validation.items()})
        score = (
            2.0 * validation['raw_code_bit_acc'] +
            4.0 * validation['post_ecc_exact_recovery'] +
            validation['crc_valid_rate'] +
            0.5 * validation['presence_tpr'] + 0.5 * validation['negative_rejection'] +
            0.03 * min(validation['incremental_watermark_psnr'], 45.0) +
            validation['incremental_watermark_ssim']
        )
        print(row)
        best_path = MODEL_DIR / f'best_v9_stage{current_stage + 1}_{config_hash}.pth'
        if score > stage_best_score:
            stage_best_score = score
            torch.save(model_snapshot(current_stage, stage_epoch, validation), best_path)
        passed_now = stage_epoch >= stage['min_epochs'] and stage_passed(stage, validation)
        if passed_now:
            stage_best_score = score
            torch.save(model_snapshot(current_stage, stage_epoch, validation), best_path)
        stage_pass_streak = stage_pass_streak + 1 if passed_now else 0
        if stage_pass_streak >= STAGE_PASS_PATIENCE:
            transition_reason = 'passed'
        elif stage_epoch >= stage['max_epochs']:
            transition_reason = 'budget_exhausted'
    else:
        print(row)
    history.append(row)

    if transition_reason is not None:
        best_path = MODEL_DIR / f'best_v9_stage{current_stage + 1}_{config_hash}.pth'
        best_stage = torch.load(best_path, map_location=device)
        latent_embedder.load_state_dict(best_stage['latent_embedder'])
        latent_extractor.load_state_dict(best_stage['latent_extractor'])
        latent_embedder.strength = best_stage['latent_strength']
        stage_records.append({
            'stage_index': current_stage + 1, 'stage_name': stage['name'],
            'transition_reason': transition_reason,
            'passed': transition_reason == 'passed', 'epochs_used': stage_epoch,
            **{f'best_{key}': value for key, value in best_stage['validation'].items()},
        })
        print(f'STAGE {transition_reason.upper()}: {stage["name"]}')
        current_stage += 1; stage_epoch = 0
        stage_best_score = -math.inf; stage_pass_streak = 0
        optimizer = None; optimizer_stage = None

    save_checkpoint(optimizer)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

best = None
for stage_index in range(len(TRAINING_STAGES) - 1, -1, -1):
    candidate = MODEL_DIR / f'best_v9_stage{stage_index + 1}_{config_hash}.pth'
    if candidate.exists():
        best = torch.load(candidate, map_location=device); break
if best is None: raise RuntimeError('Belum ada best checkpoint V9.')

latent_embedder.load_state_dict(best['latent_embedder'])
latent_extractor.load_state_dict(best['latent_extractor'])
latent_embedder.strength = best['latent_strength']
latent_embedder.eval(); latent_extractor.eval()
TRAINING_HIGHEST_STAGE = best['stage_index']
TRAINING_COMPLETED_ALL_STAGES = current_stage >= len(TRAINING_STAGES)
TRAINING_FINAL_STAGE_PASSED = any(
    record['stage_index'] == len(TRAINING_STAGES) and record['passed']
    for record in stage_records
)
pd.DataFrame(history).to_csv(REPORT_DIR / 'training_history.csv', index=False)
pd.DataFrame(stage_records).to_csv(REPORT_DIR / 'training_stage_records.csv', index=False)
print('Best stage:', TRAINING_STAGES[TRAINING_HIGHEST_STAGE]['name'])
print('Best validation:', best['validation'])
print('Seluruh tahap dijalankan:', TRAINING_COMPLETED_ALL_STAGES)


In [ ]:
# 9. Evaluasi codec, latent extraction, dan perceptual video integrity
TEST_CODEC_CONFIGS = (
    [('H264', crf) for crf in (23, 28, 35, 40)] +
    [('H265', crf) for crf in (23, 28, 35, 40)] +
    [('AV1', crf) for crf in (23, 28, 35, 40)] +
    [('NEURAL', quality) for quality in (1, 3, 5)]
)
CALIBRATION_CONFIGS = [
    ('H264', 35), ('H265', 35), ('AV1', 35), ('NEURAL', 1)
]

def codec_roundtrip(frames, fps, codec_name, level, output_base):
    output_base = Path(output_base)
    if codec_name == 'H264':
        path = output_base.with_suffix('.mp4')
        ffmpeg_encode(frames, path, fps, 'libx264', level)
        reconstructed = read_encoded_video(path, len(frames))
    elif codec_name == 'H265':
        path = output_base.with_suffix('.mp4')
        ffmpeg_encode(frames, path, fps, 'libx265', level)
        reconstructed = read_encoded_video(path, len(frames))
    elif codec_name == 'AV1':
        path = output_base.with_suffix('.mkv')
        ffmpeg_encode(frames, path, fps, 'libaom-av1', level)
        reconstructed = read_encoded_video(path, len(frames))
    elif codec_name == 'NEURAL':
        path = output_base.with_suffix('.nvc')
        plain_neural_encode(frames, path, fps, level)
        reconstructed, _ = plain_neural_decode(path)
    else:
        raise ValueError(codec_name)
    return reconstructed, path, path.stat().st_size

def mean_quality(reference, candidate):
    count = min(len(reference), len(candidate))
    psnr_values, ssim_values = [], []
    for first, second in zip(reference[:count], candidate[:count]):
        psnr_values.append(peak_signal_noise_ratio(first, second, data_range=1.0))
        ssim_values.append(structural_similarity(
            first, second, data_range=1.0, channel_axis=-1
        ))
    return float(np.mean(psnr_values)), float(np.mean(ssim_values))

def neural_baseline_roundtrip(frames, fps=25.0):
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'baseline.nvc'
        plain_neural_encode(frames, path, fps, CORE_NEURAL_QUALITY)
        reconstructed, _ = plain_neural_decode(path)
        size = path.stat().st_size
    return reconstructed, size

def make_case_frames(original, case_name, fps=25.0):
    if case_name == 'no_watermark':
        frames, size = neural_baseline_roundtrip(original, fps)
        return frames, None, size
    text = TARGET_TEXT if case_name == 'target' else CONTROL_TEXT
    payload = text_to_payload(text)
    codeword = payload_to_codeword(payload)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'watermarked.lwm'
        frames, _ = latent_watermark_encode(original, path, fps, payload)
        size = path.stat().st_size
    return frames, {
        'payload': payload.cpu().numpy().reshape(-1).astype(np.uint8),
        'codeword': codeword.cpu().numpy().reshape(-1).astype(np.uint8),
    }, size

def prediction_from_frames(frames, chunk_size=16):
    hidden_chunks = []
    with torch.inference_mode():
        tensor = frames_to_tensor(frames)
        for start in range(0, tensor.shape[0], chunk_size):
            latent = CORE_CODEC.g_a(tensor[start:start + chunk_size])
            hidden_chunks.append(latent_extractor.encode_features(latent))
        clip_hidden = torch.cat(hidden_chunks).mean(dim=0, keepdim=True)
        logits, presence_logits = latent_extractor.classify(clip_hidden)
        predicted, probabilities, presence = aggregate_decode(logits, presence_logits)
    return decode_codeword(predicted), predicted, probabilities, presence

def payload_ber(expected, predicted):
    return float(np.mean(
        np.asarray(expected).reshape(-1) != np.asarray(predicted).reshape(-1)
    ))

def codeword_ber(expected, predicted):
    return float(np.mean(
        np.asarray(expected).reshape(-1) != np.asarray(predicted).reshape(-1)
    ))

def video_fingerprint(frames, sample_count=16):
    frames = np.asarray(frames)
    indices = np.linspace(0, len(frames) - 1, min(sample_count, len(frames)), dtype=int)
    low_frequency = []
    for index in indices:
        frame = np.clip(frames[index] * 255.0, 0, 255).astype(np.uint8)
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        gray = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
        low_frequency.append(cv2.dct(gray)[:8, :8].reshape(-1))
    signature = np.median(np.stack(low_frequency), axis=0)
    threshold = np.median(signature[1:])
    bits = (signature >= threshold).astype(np.uint8)
    return bits

def fingerprint_hex(bits):
    return np.packbits(np.asarray(bits, dtype=np.uint8), bitorder='big').tobytes().hex()

def fingerprint_distance(first, second):
    return int(np.sum(
        np.asarray(first, dtype=np.uint8).reshape(-1) !=
        np.asarray(second, dtype=np.uint8).reshape(-1)
    ))

def tamper_frames(frames):
    tampered = np.asarray(frames).copy()
    height, width = tampered.shape[1:3]
    y0, y1 = height // 4, 3 * height // 4
    x0, x1 = width // 4, 3 * width // 4
    tampered[:, y0:y1, x0:x1] = 1.0 - tampered[:, y0:y1, x0:x1]
    return tampered

print('Evaluasi H.264/H.265/AV1/Neural + integrity fingerprint siap.')


In [ ]:
# 10. Kalibrasi latent watermark pada validation split
calibration_rows = []
integrity_calibration_rows = []
target_payload_np = text_to_payload(TARGET_TEXT).cpu().numpy().reshape(-1).astype(np.uint8)
control_payload_np = text_to_payload(CONTROL_TEXT).cpu().numpy().reshape(-1).astype(np.uint8)
target_codeword_np = payload_to_codeword(
    text_to_payload(TARGET_TEXT)
).cpu().numpy().reshape(-1).astype(np.uint8)

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    for video_index, path in enumerate(tqdm(VAL_PATHS, desc='Calibration')):
        original, fps = load_eval_frames(path, max_frames=32)
        original_fingerprint = video_fingerprint(original)
        for case_name in ('target', 'no_watermark', 'other_payload'):
            case_frames, own, core_bytes = make_case_frames(original, case_name, fps)
            for codec_name, level in CALIBRATION_CONFIGS:
                base = tmp / f'{video_index}_{case_name}_{codec_name}_{level}'
                reconstructed, _, _ = codec_roundtrip(case_frames, fps, codec_name, level, base)
                outcome, predicted_code, _, presence = prediction_from_frames(reconstructed)
                estimated = outcome['payload_bits'] if outcome['crc_valid'] else outcome['raw_payload_bits']
                exact_target = bool(
                    outcome['crc_valid'] and
                    np.array_equal(outcome['payload_bits'], target_payload_np)
                )
                fp_distance = fingerprint_distance(
                    original_fingerprint, video_fingerprint(reconstructed)
                )
                calibration_rows.append({
                    'video': path.name, 'case': case_name,
                    'label_target': int(case_name == 'target'),
                    'codec': codec_name, 'level': level, 'presence': presence,
                    'ecc_success': bool(outcome['ecc_success']),
                    'crc_valid': bool(outcome['crc_valid']),
                    'exact_target_payload': exact_target,
                    'BER_to_target_payload': payload_ber(target_payload_np, estimated),
                    'raw_BER_to_target_codeword': codeword_ber(target_codeword_np, predicted_code),
                    'raw_own_payload_bit_acc': np.nan if own is None else 1.0 - payload_ber(
                        own['payload'], outcome['raw_payload_bits']
                    ),
                    'post_ecc_own_exact': False if own is None else bool(
                        outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], own['payload'])
                    ),
                    'decoded_text': outcome['decoded_text'],
                    'fingerprint_distance': fp_distance,
                    'integrity_valid': fp_distance <= INTEGRITY_MAX_DISTANCE,
                    'core_bitstream_bytes': core_bytes,
                })
                if case_name == 'target':
                    integrity_calibration_rows.append({
                        'video': path.name, 'codec': codec_name, 'level': level,
                        'case': 'valid_target', 'expected_integrity_label': 1,
                        'fingerprint_distance': fp_distance,
                    })
                    tampered = tamper_frames(case_frames)
                    tampered_base = tmp / f'{video_index}_tampered_{codec_name}_{level}'
                    tampered_reconstructed, _, _ = codec_roundtrip(
                        tampered, fps, codec_name, level, tampered_base
                    )
                    integrity_calibration_rows.append({
                        'video': path.name, 'codec': codec_name, 'level': level,
                        'case': 'tampered_target', 'expected_integrity_label': 0,
                        'fingerprint_distance': fingerprint_distance(
                            original_fingerprint, video_fingerprint(tampered_reconstructed)
                        ),
                    })

calibration_df = pd.DataFrame(calibration_rows)
integrity_calibration_df = pd.DataFrame(integrity_calibration_rows)

def calibrate_presence_threshold(frame, max_fpr=0.05):
    labels = frame['label_target'].to_numpy().astype(bool)
    no_watermark = frame['case'].to_numpy() == 'no_watermark'
    other_payload = frame['case'].to_numpy() == 'other_payload'
    exact = frame['exact_target_payload'].to_numpy().astype(bool)
    best_choice = None
    for threshold in np.r_[np.linspace(0.05, 0.995, 40), 1.001]:
        predicted = (frame['presence'].to_numpy() >= threshold) & exact
        tp = int(np.sum(predicted & labels)); fn = int(np.sum(~predicted & labels))
        fp = int(np.sum(predicted & ~labels)); tn = int(np.sum(~predicted & ~labels))
        tpr = tp / max(tp + fn, 1); fpr = fp / max(fp + tn, 1)
        no_wm_fpr = predicted[no_watermark].mean()
        other_fpr = predicted[other_payload].mean()
        if no_wm_fpr <= max_fpr and other_fpr <= max_fpr:
            candidate = (tpr, -max(no_wm_fpr, other_fpr), -fpr, threshold)
            if best_choice is None or candidate > best_choice[0]:
                best_choice = (candidate, threshold, tpr, fpr, no_wm_fpr, other_fpr)
    return best_choice[1:]

(PRESENCE_THRESHOLD, val_tpr, val_fpr,
 calibrated_no_wm_fpr, calibrated_other_fpr) = calibrate_presence_threshold(calibration_df)

def calibrate_integrity_threshold(frame, max_tamper_fpr=0.05):
    labels = frame['expected_integrity_label'].to_numpy().astype(bool)
    distances = frame['fingerprint_distance'].to_numpy()
    best_choice = None
    for threshold in range(-1, 65):
        predicted_valid = distances <= threshold
        tp = int(np.sum(predicted_valid & labels))
        fn = int(np.sum(~predicted_valid & labels))
        fp = int(np.sum(predicted_valid & ~labels))
        tn = int(np.sum(~predicted_valid & ~labels))
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        if fpr <= max_tamper_fpr:
            candidate = (tpr, -fpr, threshold)
            if best_choice is None or candidate > best_choice[0]:
                best_choice = (candidate, threshold, tpr, fpr)
    if best_choice is None:
        raise RuntimeError('Tidak ada ambang integritas yang memenuhi batas false accept.')
    return best_choice[1:]

INTEGRITY_MAX_DISTANCE, integrity_val_tpr, integrity_val_fpr = calibrate_integrity_threshold(
    integrity_calibration_df
)
calibration_df['detected_as_sabila'] = (
    (calibration_df['presence'] >= PRESENCE_THRESHOLD) &
    calibration_df['exact_target_payload']
)
calibration_df['integrity_valid'] = (
    calibration_df['fingerprint_distance'] <= INTEGRITY_MAX_DISTANCE
)

target_rows = calibration_df.query("case == 'target'")
other_rows = calibration_df.query("case == 'other_payload'")
target_exact = target_rows['post_ecc_own_exact'].mean()
other_exact = other_rows['post_ecc_own_exact'].mean()
no_wm_fpr = calibration_df.query("case == 'no_watermark'")['detected_as_sabila'].mean()
other_target_fpr = other_rows['detected_as_sabila'].mean()
integrity_calibration_df['integrity_valid'] = (
    integrity_calibration_df['fingerprint_distance'] <= INTEGRITY_MAX_DISTANCE
)

diagnostic_df = calibration_df.groupby(['codec', 'level', 'case']).agg(
    presence_mean=('presence', 'mean'), crc_valid_rate=('crc_valid', 'mean'),
    post_ecc_exact_rate=('post_ecc_own_exact', 'mean'),
    fingerprint_distance_mean=('fingerprint_distance', 'mean'),
    integrity_valid_rate=('integrity_valid', 'mean'),
    detected_as_sabila_rate=('detected_as_sabila', 'mean'),
).reset_index()

print(f'PRESENCE_THRESHOLD={PRESENCE_THRESHOLD:.3f}')
print(f'Validation TPR/FPR: {val_tpr:.3f}/{val_fpr:.3f}')
print(f'Exact sabila/control: {target_exact:.3f}/{other_exact:.3f}')
print(f'No-watermark/other-payload FPR: {no_wm_fpr:.3f}/{other_target_fpr:.3f}')
print(
    f'Integrity threshold={INTEGRITY_MAX_DISTANCE}; '
    f'validation TPR/FPR={integrity_val_tpr:.3f}/{integrity_val_fpr:.3f}'
)
display(diagnostic_df)

gate_passed = (
    TRAINING_COMPLETED_ALL_STAGES and
    TRAINING_FINAL_STAGE_PASSED and
    TRAINING_HIGHEST_STAGE == len(TRAINING_STAGES) - 1 and
    val_tpr >= 0.80 and val_fpr <= 0.05 and
    target_exact >= 0.80 and other_exact >= 0.80 and
    no_wm_fpr <= 0.05 and other_target_fpr <= 0.05 and
    integrity_val_tpr >= 0.90 and integrity_val_fpr <= 0.05
)
calibration_df.to_csv(REPORT_DIR / 'validation_calibration.csv', index=False)
integrity_calibration_df.to_csv(
    REPORT_DIR / 'validation_integrity_calibration.csv', index=False
)
diagnostic_df.to_csv(REPORT_DIR / 'validation_diagnostic_by_codec.csv', index=False)
with (REPORT_DIR / 'calibrated_thresholds.json').open('w') as handle:
    json.dump({
        'presence_threshold': PRESENCE_THRESHOLD,
        'integrity_max_fingerprint_distance': INTEGRITY_MAX_DISTANCE,
        'reed_solomon_required': True, 'crc16_required': True,
        'validation_tpr': val_tpr, 'validation_fpr': val_fpr,
        'integrity_validation_tpr': integrity_val_tpr,
        'integrity_validation_fpr': integrity_val_fpr,
    }, handle, indent=2)
print('VALIDATION GATE:', 'LULUS' if gate_passed else 'GAGAL — final test hanya diagnostik')


In [ ]:
# 11. Final test: latent watermark, codec attacks, dan tamper controls
detailed_rows, imperceptibility_rows, fingerprint_manifest = [], [], []

for video_index, path in enumerate(tqdm(TEST_PATHS, desc='Final test')):
    original, fps = load_eval_frames(path)
    original_fingerprint = video_fingerprint(original)
    safe_stem = re.sub(r'[^A-Za-z0-9_.-]+', '_', path.stem)

    baseline_path = BITSTREAM_DIR / f'{safe_stem}_core_baseline.nvc'
    plain_neural_encode(original, baseline_path, fps, CORE_NEURAL_QUALITY)
    baseline_frames, _ = plain_neural_decode(baseline_path)

    target_payload = text_to_payload(TARGET_TEXT)
    target_core_path = BITSTREAM_DIR / f'{safe_stem}_target_latent.lwm'
    target_frames, _ = latent_watermark_encode(
        original, target_core_path, fps, target_payload
    )
    target_own = {
        'payload': target_payload.cpu().numpy().reshape(-1).astype(np.uint8),
        'codeword': payload_to_codeword(target_payload).cpu().numpy().reshape(-1).astype(np.uint8),
    }

    control_payload = text_to_payload(CONTROL_TEXT)
    control_core_path = BITSTREAM_DIR / f'{safe_stem}_control_latent.lwm'
    control_frames, _ = latent_watermark_encode(
        original, control_core_path, fps, control_payload
    )
    control_own = {
        'payload': control_payload.cpu().numpy().reshape(-1).astype(np.uint8),
        'codeword': payload_to_codeword(control_payload).cpu().numpy().reshape(-1).astype(np.uint8),
    }
    tampered_frames = tamper_frames(target_frames)

    original_to_baseline_psnr, original_to_baseline_ssim = mean_quality(
        original, baseline_frames
    )
    original_to_target_psnr, original_to_target_ssim = mean_quality(
        original, target_frames
    )
    incremental_psnr, incremental_ssim = mean_quality(baseline_frames, target_frames)
    raw_bytes = len(original) * original.shape[1] * original.shape[2] * 3
    imperceptibility_rows.append({
        'video': path.name,
        'PSNR_original_vs_neural_baseline': original_to_baseline_psnr,
        'SSIM_original_vs_neural_baseline': original_to_baseline_ssim,
        'PSNR_original_vs_latent_watermarked': original_to_target_psnr,
        'SSIM_original_vs_latent_watermarked': original_to_target_ssim,
        'PSNR_neural_baseline_vs_latent_watermarked': incremental_psnr,
        'SSIM_neural_baseline_vs_latent_watermarked': incremental_ssim,
        'latent_core_bitstream_bytes': target_core_path.stat().st_size,
        'latent_core_compression_factor': raw_bytes / max(target_core_path.stat().st_size, 1),
    })
    fingerprint_manifest.append({
        'video': path.name, 'video_stem': path.stem,
        'reference_fingerprint_hex': fingerprint_hex(original_fingerprint),
    })

    cases = {
        'target': (target_frames, target_own, target_core_path.stat().st_size, 1),
        'no_watermark': (baseline_frames, None, baseline_path.stat().st_size, 1),
        'other_payload': (control_frames, control_own, control_core_path.stat().st_size, 1),
        'tampered_target': (tampered_frames, target_own, target_core_path.stat().st_size, 0),
    }

    for case_name, (case_frames, own, core_bytes, expected_integrity) in cases.items():
        for codec_name, level in TEST_CODEC_CONFIGS:
            output_base = BITSTREAM_DIR / f'{safe_stem}_{case_name}_{codec_name.lower()}_{level}'
            reconstructed, bitstream_path, encoded_bytes = codec_roundtrip(
                case_frames, fps, codec_name, level, output_base
            )
            outcome, predicted_code, _, presence = prediction_from_frames(reconstructed)
            estimated = outcome['payload_bits'] if outcome['crc_valid'] else outcome['raw_payload_bits']
            exact_target = bool(
                outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], target_payload_np)
            )
            detected = presence >= PRESENCE_THRESHOLD and exact_target
            fp_distance = fingerprint_distance(
                original_fingerprint, video_fingerprint(reconstructed)
            )
            integrity_valid = fp_distance <= INTEGRITY_MAX_DISTANCE
            quality_psnr, quality_ssim = mean_quality(case_frames, reconstructed)
            n, h, w, _ = case_frames.shape
            duration = n / fps
            attack_raw_bytes = n * h * w * 3
            detailed_rows.append({
                'video': path.name, 'split': 'test', 'case': case_name,
                'true_target_label': int(case_name in ('target', 'tampered_target')),
                'expected_integrity_label': int(expected_integrity),
                'codec': codec_name, 'level': int(level), 'frames': n,
                'resolution': f'{w}x{h}', 'duration_s': duration,
                'core_latent_bitstream_bytes': core_bytes,
                'attack_encoded_bytes': encoded_bytes,
                'bpp': encoded_bytes * 8 / (n * h * w),
                'bitrate_kbps': encoded_bytes * 8 / max(duration, 1e-9) / 1000,
                'compression_factor_raw_over_encoded': attack_raw_bytes / max(encoded_bytes, 1),
                'PSNR_input_vs_reconstructed': quality_psnr,
                'SSIM_input_vs_reconstructed': quality_ssim,
                'presence_probability': presence,
                'ecc_success': bool(outcome['ecc_success']),
                'crc_valid': bool(outcome['crc_valid']),
                'corrected_symbols': outcome['corrected_symbols'],
                'BER_to_sabila_48_payload_bits': payload_ber(target_payload_np, estimated),
                'raw_BER_to_sabila_codeword': codeword_ber(target_codeword_np, predicted_code),
                'detected_as_sabila': bool(detected),
                'decoded_text': outcome['decoded_text'],
                'raw_text_before_ecc': outcome['raw_text'],
                'post_ecc_own_exact': False if own is None else bool(
                    outcome['crc_valid'] and np.array_equal(outcome['payload_bits'], own['payload'])
                ),
                'fingerprint_distance_to_original': fp_distance,
                'integrity_valid': bool(integrity_valid),
                'automatic_verification_passed': bool(detected and integrity_valid),
                'attack_bitstream_file': str(bitstream_path),
            })

detailed_df = pd.DataFrame(detailed_rows)
imperceptibility_df = pd.DataFrame(imperceptibility_rows)
fingerprint_manifest_df = pd.DataFrame(fingerprint_manifest)
fingerprint_manifest_df.to_csv(REPORT_DIR / 'fingerprint_manifest.csv', index=False)
print('Final test selesai:', len(detailed_df), 'sampel termasuk tamper controls.')


In [ ]:
# 12. Detection, recovery, integrity, quality, dan acceptance report
def confusion_metrics(group, truth_column, prediction_column):
    truth = group[truth_column].astype(bool).to_numpy()
    pred = group[prediction_column].astype(bool).to_numpy()
    tp = int(np.sum(truth & pred)); fn = int(np.sum(truth & ~pred))
    fp = int(np.sum(~truth & pred)); tn = int(np.sum(~truth & ~pred))
    return {
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'accuracy': (tp + tn) / max(tp + tn + fp + fn, 1),
        'precision': tp / max(tp + fp, 1),
        'recall_TPR': tp / max(tp + fn, 1),
        'specificity_TNR': tn / max(tn + fp, 1),
        'FPR': fp / max(fp + tn, 1),
    }

detection_df = detailed_df.query("case != 'tampered_target'")
summary_rows = []
for (codec_name, level), group in detection_df.groupby(['codec', 'level'], sort=False):
    summary_rows.append({
        'codec': codec_name, 'level': level,
        **confusion_metrics(group, 'true_target_label', 'detected_as_sabila'),
    })
summary_df = pd.DataFrame(summary_rows)

target_recovery = detection_df.query("case == 'target'").groupby(
    ['codec', 'level']
)['post_ecc_own_exact'].mean().rename('exact_sabila_recovery').reset_index()
control_recovery = detection_df.query("case == 'other_payload'").groupby(
    ['codec', 'level']
)['post_ecc_own_exact'].mean().rename('exact_control_recovery').reset_index()
compression_summary = detection_df.query("case == 'target'").groupby(
    ['codec', 'level']
)[['bpp', 'bitrate_kbps', 'compression_factor_raw_over_encoded',
   'PSNR_input_vs_reconstructed', 'SSIM_input_vs_reconstructed']].mean().reset_index()
summary_df = summary_df.merge(target_recovery, on=['codec', 'level']).merge(
    control_recovery, on=['codec', 'level']
).merge(compression_summary, on=['codec', 'level'])

integrity_df = detailed_df.query("case in ['target', 'tampered_target']")
integrity_metrics = confusion_metrics(
    integrity_df, 'expected_integrity_label', 'integrity_valid'
)
tamper_rejection = 1.0 - integrity_df.query(
    "case == 'tampered_target'"
)['integrity_valid'].mean()

high_compression = summary_df[
    ((summary_df.codec.isin(['H264', 'H265', 'AV1'])) & (summary_df.level >= 35)) |
    ((summary_df.codec == 'NEURAL') & (summary_df.level == 1))
]
acceptance = {
    'latent_embedding_used': True,
    'split_disjoint': True,
    'all_training_stages_executed': bool(TRAINING_COMPLETED_ALL_STAGES),
    'final_training_stage_passed': bool(TRAINING_FINAL_STAGE_PASSED),
    'validation_gate_passed': bool(gate_passed),
    'test_FPR_at_most_5_percent': bool((summary_df['FPR'] <= 0.05).all()),
    'high_compression_recall_at_least_80_percent': bool(
        (high_compression['recall_TPR'] >= 0.80).all()
    ),
    'control_payload_recovery_at_least_80_percent': bool(
        (summary_df['exact_control_recovery'] >= 0.80).all()
    ),
    'integrity_accuracy_at_least_90_percent': integrity_metrics['accuracy'] >= 0.90,
    'tamper_rejection_at_least_90_percent': tamper_rejection >= 0.90,
    'latent_bitstream_smaller_than_raw': bool(
        (imperceptibility_df['latent_core_compression_factor'] > 1.0).all()
    ),
    'incremental_watermark_PSNR_at_least_34dB': bool(
        imperceptibility_df['PSNR_neural_baseline_vs_latent_watermarked'].mean() >= 34.0
    ),
    'incremental_watermark_SSIM_at_least_0_94': bool(
        imperceptibility_df['SSIM_neural_baseline_vs_latent_watermarked'].mean() >= 0.94
    ),
}
acceptance['all_passed'] = all(acceptance.values())

detailed_df.to_csv(REPORT_DIR / 'test_detailed.csv', index=False)
summary_df.to_csv(REPORT_DIR / 'test_summary.csv', index=False)
imperceptibility_df.to_csv(REPORT_DIR / 'imperceptibility.csv', index=False)
with (REPORT_DIR / 'integrity_metrics.json').open('w') as handle:
    json.dump({**integrity_metrics, 'tamper_rejection': tamper_rejection}, handle, indent=2)
with (REPORT_DIR / 'acceptance_report.json').open('w') as handle:
    json.dump(acceptance, handle, indent=2)

display(summary_df)
display(imperceptibility_df.describe())
print('\nINTEGRITY METRICS:', integrity_metrics, 'tamper_rejection=', tamper_rejection)
print('\nACCEPTANCE REPORT')
for criterion, passed in acceptance.items():
    print(f"{'LULUS' if passed else 'GAGAL'} | {criterion}")
print(
    '\nSELURUH KRITERIA LULUS.' if acceptance['all_passed'] else
    '\nEksperimen selesai, tetapi hasil belum mendukung seluruh klaim.'
)


In [ ]:
# 13. Visualisasi ringkas
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

labels = summary_df['codec'] + '-' + summary_df['level'].astype(str)
axes[0].bar(labels, summary_df['recall_TPR'])
axes[0].axhline(0.80, color='red', linestyle='--')
axes[0].set_title('Recall latent watermark setelah RS/CRC')
axes[0].tick_params(axis='x', rotation=60)
axes[0].set_ylim(0, 1.05)

axes[1].bar(labels, summary_df['FPR'])
axes[1].axhline(0.05, color='red', linestyle='--')
axes[1].set_title('False Positive Rate')
axes[1].tick_params(axis='x', rotation=60)
axes[1].set_ylim(0, max(0.1, summary_df['FPR'].max() * 1.2))

axes[2].bar(labels, summary_df['bpp'])
axes[2].set_title('Bit per pixel dari bitstream nyata')
axes[2].tick_params(axis='x', rotation=60)

plt.tight_layout()
figure_path = REPORT_DIR / 'validated_summary.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('Laporan tersimpan di:', REPORT_DIR)


In [ ]:
# 14. Tool verifikasi otomatis: upload → extract → integrity → report
def uploaded_path(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return value.get('path') or value.get('video') or value.get('name')
    return str(value)

def verify_uploaded_video(video_upload, reference_upload, expected_text):
    video_path = uploaded_path(video_upload)
    reference_path = uploaded_path(reference_upload)
    if not video_path:
        return {'status': 'ERROR', 'message': 'Video yang diverifikasi belum dipilih.'}
    try:
        expected_payload = text_to_payload(expected_text)
    except Exception as error:
        return {'status': 'ERROR', 'message': str(error)}

    frames, _ = load_eval_frames(Path(video_path))
    outcome, _, _, presence = prediction_from_frames(frames)
    expected_bits = expected_payload.cpu().numpy().reshape(-1).astype(np.uint8)
    watermark_verified = bool(
        presence >= PRESENCE_THRESHOLD and outcome['crc_valid'] and
        np.array_equal(outcome['payload_bits'], expected_bits)
    )

    fingerprint_distance_value = None
    integrity_valid = None
    integrity_status = 'REFERENCE_REQUIRED'
    if reference_path:
        reference_frames, _ = load_eval_frames(Path(reference_path))
        fingerprint_distance_value = fingerprint_distance(
            video_fingerprint(reference_frames), video_fingerprint(frames)
        )
        integrity_valid = fingerprint_distance_value <= INTEGRITY_MAX_DISTANCE
        integrity_status = 'VALID' if integrity_valid else 'MODIFIED'

    if watermark_verified and integrity_valid is True:
        overall_status = 'VERIFIED'
    elif watermark_verified and integrity_valid is None:
        overall_status = 'WATERMARK_VALID_INTEGRITY_NOT_CHECKED'
    elif watermark_verified:
        overall_status = 'WATERMARK_VALID_BUT_CONTENT_MODIFIED'
    else:
        overall_status = 'WATERMARK_INVALID_OR_NOT_FOUND'

    report = {
        'status': overall_status,
        'expected_text': expected_text,
        'decoded_text': outcome['decoded_text'],
        'raw_text_before_ecc': outcome['raw_text'],
        'presence_probability': round(float(presence), 6),
        'presence_threshold': round(float(PRESENCE_THRESHOLD), 6),
        'ecc_success': bool(outcome['ecc_success']),
        'crc_valid': bool(outcome['crc_valid']),
        'corrected_symbols': None if np.isnan(outcome['corrected_symbols']) else int(outcome['corrected_symbols']),
        'watermark_verified': watermark_verified,
        'integrity_status': integrity_status,
        'fingerprint_distance': fingerprint_distance_value,
        'integrity_max_distance': INTEGRITY_MAX_DISTANCE,
        'integrity_valid': integrity_valid,
        'backbone': f'bmshj2018_factorized_quality_{CORE_NEURAL_QUALITY}_framewise',
    }
    with (REPORT_DIR / 'last_upload_verification.json').open('w') as handle:
        json.dump(report, handle, indent=2)
    return report

verification_app = gr.Interface(
    fn=verify_uploaded_video,
    inputs=[
        gr.Video(label='Video yang diverifikasi', sources=['upload']),
        gr.Video(label='Video referensi untuk integritas (opsional)', sources=['upload']),
        gr.Textbox(value=TARGET_TEXT, label='Watermark yang diharapkan'),
    ],
    outputs=gr.JSON(label='Laporan verifikasi otomatis'),
    title='Latent Neural Watermark Verification',
    description=(
        'Mengekstrak watermark melalui latent neural codec, memvalidasi RS/CRC, '
        'dan membandingkan perceptual fingerprint jika video referensi diberikan.'
    ),
)
verification_app.launch(share=True, debug=False)


## Cara membaca hasil V9

Pipeline ini memisahkan tiga klaim yang berbeda:

1. **Latent embedding:** file `.lwm` membuktikan watermark disisipkan setelah neural
   analysis transform dan sebelum entropy coding/neural decoder.
2. **Watermark verification:** presence harus melewati threshold, Reed–Solomon berhasil,
   CRC-16 valid, dan payload harus tepat.
3. **Video integrity:** perceptual fingerprint video upload dibandingkan dengan video
   referensi. CRC payload tidak dipakai sebagai pengganti validasi isi video.

Klaim tahan kompresi tinggi hanya boleh digunakan jika validation gate dan seluruh
acceptance report lulus untuk H.264, H.265/HEVC, AV1, serta neural recompression.

V8 pixel-domain dapat dipakai sebagai baseline pembanding. V9 merupakan algoritma utama
latent-domain. Backbone saat ini masih frame-wise; tuliskan batasan tersebut secara eksplisit
jika belum diganti dengan temporal neural video codec.
